In [1]:
import pandas as pd

# 1. Load raw file
df_raw = pd.read_excel("Cactus 2025.xlsx")

# 2. Promote the first row to header
header = df_raw.iloc[0]
df = df_raw.iloc[1:].copy()
df.columns = header

In [2]:
df.head(10)

,/@count,/@count/#agg,/@found,/@found/#agg,/@name,/@start,/@start/#agg,/transaction/#id,/transaction/colour,/transaction/colour/#agg,...,/transaction/subfile/detail/detail.tax/#agg,/transaction/subfile/detail/detail.taxcode,/transaction/subfile/detail/detail.unitprice,/transaction/subfile/detail/detail.unitprice/#agg,/transaction/theirref,/transaction/tofrom,/transaction/transdate,/transaction/transdate/#agg,/transaction/type,/transaction/user8
1,1030,1030,1030,1030,Transaction,0,0,1,2,2,...,NaN,*,38,38,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,20251203,20251203,DII,NaN
2,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,34,34,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,20251203,DII,NaN
3,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,155,155,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,NaN,DII,NaN
4,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,19,19,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,NaN,DII,NaN
5,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,23.5,23.5,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,NaN,DII,NaN
6,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,19.99,19.99,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,NaN,DII,NaN
7,1030,NaN,1030,NaN,Transaction,0,NaN,2,NaN,NaN,...,NaN,*,29.99,29.99,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),20251203,NaN,DII,NaN
8,1030,NaN,1030,NaN,Transaction,0,NaN,3,NaN,NaN,...,NaN,*,59,59,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,20251203,20251203,DII,NaN
9,1030,NaN,1030,NaN,Transaction,0,NaN,3,NaN,NaN,...,NaN,*,10.5,10.5,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,20251203,NaN,DII,NaN
10,1030,NaN,1030,NaN,Transaction,0,NaN,3,NaN,NaN,...,NaN,*,38,38,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,20251203,NaN,DII,NaN


In [3]:
# 3. Drop aggregate/meta columns we don’t need
cols_to_drop = [
    c for c in df.columns
    if c.startswith("/@")           # meta like /@count, /@found
    or c.endswith("#agg")           # columns ending in #agg
    or c == "/transaction/subfile/@name"
]
df = df.drop(columns=cols_to_drop)


In [4]:
# 4. Convert key date fields
date_cols = [
    "/transaction/transdate",
    "/transaction/duedate",
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col].astype(str), format="%Y%m%d", errors="coerce")

In [5]:
# 5. Convert numeric fields
num_cols = [
    "/transaction/gross",
    "/transaction/subfile/detail/detail.net",
    "/transaction/subfile/detail/detail.tax",
    "/transaction/subfile/detail/detail.discount",
    "/transaction/subfile/detail/detail.orderqty",
    "/transaction/subfile/detail/detail.costprice",
    "/transaction/subfile/detail/detail.unitprice",
    "/transaction/subfile/detail/detail.stockqty",
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
# 6. Clean up key string fields
str_cols = [
    "/transaction/tofrom",
    "/transaction/deliveryaddress",
    "/transaction/description",
    "/transaction/subfile/detail/detail.description",
    "/transaction/subfile/detail/detail.stockcode",
    "/transaction/paymentmethod",
]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

In [7]:
df.head(10)

,/transaction/#id,/transaction/colour,/transaction/contra,/transaction/deliveryaddress,/transaction/description,/transaction/duedate,/transaction/gross,/transaction/mailingaddress,/transaction/namecode,/transaction/ourref,...,/transaction/subfile/detail/detail.stockcode,/transaction/subfile/detail/detail.stockqty,/transaction/subfile/detail/detail.tax,/transaction/subfile/detail/detail.taxcode,/transaction/subfile/detail/detail.unitprice,/transaction/theirref,/transaction/tofrom,/transaction/transdate,/transaction/type,/transaction/user8
1,1,2,1500,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE 77...,CR FOR 166080,2025-12-18,-38.00,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE\n1...,*CACTUSCLUB,166133,...,TOM2420,-1.0,NaN,*,38.00,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,2025-12-03,DII,NaN
2,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,PRO2302,6.0,NaN,*,34.00,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
3,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,BUT2010,1.0,NaN,*,155.00,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
4,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,ONI2190,1.0,NaN,*,19.00,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
5,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,ONI2270,1.0,NaN,*,23.50,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
6,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,GRO2802,1.0,NaN,*,19.99,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
7,2,NaN,1500,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,<NA>,2025-12-18,451.48,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDE...,*CACTUSSHER,166131,...,GRO2275,1.0,NaN,*,29.99,DEC 03 2025,THE CACTUS CLUB CAFE (3100 LTD - SHERWAY GARDENS),2025-12-03,DII,NaN
8,3,NaN,1500,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE 77...,<NA>,2025-12-18,761.30,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE&#1...,*CACTUSCLUB,166080,...,BRO2000,5.0,NaN,*,59.00,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,2025-12-03,DII,NaN
9,3,NaN,1500,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE 77...,<NA>,2025-12-18,761.30,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE&#1...,*CACTUSCLUB,166080,...,CUC2050,4.0,NaN,*,10.50,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,2025-12-03,DII,NaN
10,3,NaN,1500,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE 77...,<NA>,2025-12-18,761.30,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE&#1...,*CACTUSCLUB,166080,...,CUC2080,1.0,NaN,*,38.00,DEC 03 2025,THE CACTUS CLUB CAFE - FIRST CANADIAN PLACE,2025-12-03,DII,NaN


In [8]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop any column that contains the word "address"
df = df.loc[:, ~df.columns.str.contains("address", case=False)]

In [10]:
df = df.drop('/transaction/tofrom', axis=1)

In [11]:
df.head(10)

,/transaction/#id,/transaction/colour,/transaction/contra,/transaction/description,/transaction/duedate,/transaction/gross,/transaction/namecode,/transaction/ourref,/transaction/paymentmethod,/transaction/prodpricecode,...,/transaction/subfile/detail/detail.saleunit,/transaction/subfile/detail/detail.stockcode,/transaction/subfile/detail/detail.stockqty,/transaction/subfile/detail/detail.tax,/transaction/subfile/detail/detail.taxcode,/transaction/subfile/detail/detail.unitprice,/transaction/theirref,/transaction/transdate,/transaction/type,/transaction/user8
1,1,2,1500,CR FOR 166080,2025-12-18,-38.00,*CACTUSCLUB,166133,2,E,...,CS,TOM2420,-1.0,NaN,*,38.00,DEC 03 2025,2025-12-03,DII,NaN
2,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,PRO2302,6.0,NaN,*,34.00,DEC 03 2025,2025-12-03,DII,NaN
3,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,BUT2010,1.0,NaN,*,155.00,DEC 03 2025,2025-12-03,DII,NaN
4,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,ONI2190,1.0,NaN,*,19.00,DEC 03 2025,2025-12-03,DII,NaN
5,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,ONI2270,1.0,NaN,*,23.50,DEC 03 2025,2025-12-03,DII,NaN
6,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,BG,GRO2802,1.0,NaN,*,19.99,DEC 03 2025,2025-12-03,DII,NaN
7,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,BG,GRO2275,1.0,NaN,*,29.99,DEC 03 2025,2025-12-03,DII,NaN
8,3,NaN,1500,<NA>,2025-12-18,761.30,*CACTUSCLUB,166080,<NA>,E,...,CS,BRO2000,5.0,NaN,*,59.00,DEC 03 2025,2025-12-03,DII,NaN
9,3,NaN,1500,<NA>,2025-12-18,761.30,*CACTUSCLUB,166080,<NA>,E,...,CS,CUC2050,4.0,NaN,*,10.50,DEC 03 2025,2025-12-03,DII,NaN
10,3,NaN,1500,<NA>,2025-12-18,761.30,*CACTUSCLUB,166080,<NA>,E,...,CS,CUC2080,1.0,NaN,*,38.00,DEC 03 2025,2025-12-03,DII,NaN


In [12]:
# 7. Ensure transaction ID is clean
df["/transaction/#id"] = pd.to_numeric(df["/transaction/#id"], errors="coerce").astype("Int64")

# At this point, df is your clean “fact table” of line items

In [13]:
df.head()

,/transaction/#id,/transaction/colour,/transaction/contra,/transaction/description,/transaction/duedate,/transaction/gross,/transaction/namecode,/transaction/ourref,/transaction/paymentmethod,/transaction/prodpricecode,...,/transaction/subfile/detail/detail.saleunit,/transaction/subfile/detail/detail.stockcode,/transaction/subfile/detail/detail.stockqty,/transaction/subfile/detail/detail.tax,/transaction/subfile/detail/detail.taxcode,/transaction/subfile/detail/detail.unitprice,/transaction/theirref,/transaction/transdate,/transaction/type,/transaction/user8
1,1,2,1500,CR FOR 166080,2025-12-18,-38.00,*CACTUSCLUB,166133,2,E,...,CS,TOM2420,-1.0,NaN,*,38.0,DEC 03 2025,2025-12-03,DII,NaN
2,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,PRO2302,6.0,NaN,*,34.0,DEC 03 2025,2025-12-03,DII,NaN
3,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,BUT2010,1.0,NaN,*,155.0,DEC 03 2025,2025-12-03,DII,NaN
4,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,ONI2190,1.0,NaN,*,19.0,DEC 03 2025,2025-12-03,DII,NaN
5,2,NaN,1500,<NA>,2025-12-18,451.48,*CACTUSSHER,166131,2,A,...,CS,ONI2270,1.0,NaN,*,23.5,DEC 03 2025,2025-12-03,DII,NaN


Profiling Pipeline – Understanding Data Quality & Shape

The goal of profiling is to answer:

How complete is each column (null %)?

How many distinct values?

Are there obvious outliers?

Are there columns that look “broken” (e.g. all one value, weird types)?

In [14]:
import numpy as np

def profile_column(s: pd.Series) -> dict:
    out = {
        "dtype": s.dtype,
        "non_null_count": int(s.notna().sum()),
        "null_pct": float(s.isna().mean()),
        "distinct_count": int(s.nunique(dropna=True)),
    }
    if pd.api.types.is_numeric_dtype(s):
        out.update({
            "min": float(s.min()) if s.notna().any() else None,
            "max": float(s.max()) if s.notna().any() else None,
            "mean": float(s.mean()) if s.notna().any() else None,
        })
    return out

profile = {col: profile_column(df[col]) for col in df.columns}
profile_df = pd.DataFrame(profile).T.sort_values("null_pct", ascending=False)

# Optional: save to Excel/CSV to inspect
profile_df.to_excel("Cactus_2025_profiling_summary.xlsx")

In [15]:
rule_gross_vs_net = (
    df.groupby("/transaction/#id")
      .agg(
          gross=("/transaction/gross", "first"),
          net_sum=("/transaction/subfile/detail/detail.net", "sum")
      )
)

rule_gross_vs_net["diff"] = rule_gross_vs_net["gross"].round(2) - rule_gross_vs_net["net_sum"].round(2)
rule_gross_vs_net["rule_pass"] = rule_gross_vs_net["diff"].abs() < 0.01

gross_rule_failure_rate = 1 - rule_gross_vs_net["rule_pass"].mean()

In [17]:
critical_cols = [
    "/transaction/#id",
    "/transaction/transdate",
    "/transaction/duedate",
    "/transaction/subfile/detail/detail.stockcode",
    "/transaction/subfile/detail/detail.net",
]
critical_completeness = {
    col: 1 - df[col].isna().mean()
    for col in critical_cols
}

In [ ]:
df["month"] = df["/transaction/transdate"].dt.to_period("M")

In [25]:
print(df.columns.tolist())

['/transaction/#id', '/transaction/colour', '/transaction/contra', '/transaction/description', '/transaction/duedate', '/transaction/gross', '/transaction/namecode', '/transaction/ourref', '/transaction/paymentmethod', '/transaction/prodpricecode', '/transaction/subfile/detail/detail.account', '/transaction/subfile/detail/detail.costprice', '/transaction/subfile/detail/detail.description', '/transaction/subfile/detail/detail.discount', '/transaction/subfile/detail/detail.gross', '/transaction/subfile/detail/detail.net', '/transaction/subfile/detail/detail.orderqty', '/transaction/subfile/detail/detail.saleunit', '/transaction/subfile/detail/detail.stockcode', '/transaction/subfile/detail/detail.stockqty', '/transaction/subfile/detail/detail.tax', '/transaction/subfile/detail/detail.taxcode', '/transaction/subfile/detail/detail.unitprice', '/transaction/theirref', '/transaction/transdate', '/transaction/type', '/transaction/user8', 'month']


In [22]:
print(df[["/transaction/transdate", "month"]].head())

0 /transaction/transdate    month
1             2025-12-03  2025-12
2             2025-12-03  2025-12
3             2025-12-03  2025-12
4             2025-12-03  2025-12
5             2025-12-03  2025-12


In [19]:
rule_gross_vs_net_reset = rule_gross_vs_net.reset_index()
df_tx = df[["/transaction/#id", "month"]].drop_duplicates("/transaction/#id")
df_rules = df_tx.merge(rule_gross_vs_net_reset[["/transaction/#id", "rule_pass"]], on="/transaction/#id", how="left")

In [26]:
df_rules

,/transaction/#id,month,rule_pass
0,1,2025-12,True
1,2,2025-12,True
2,3,2025-12,True
3,4,2025-12,False
4,5,2025-12,False
...,...,...,...
1025,1026,2025-01,True
1026,1027,2025-01,True
1027,1028,2025-01,True
1028,1029,2025-01,False


In [27]:
merged = df.merge(df_rules, on="/transaction/#id", how="left")

print(merged.columns)

Index(['/transaction/#id', '/transaction/colour', '/transaction/contra',
       '/transaction/description', '/transaction/duedate',
       '/transaction/gross', '/transaction/namecode', '/transaction/ourref',
       '/transaction/paymentmethod', '/transaction/prodpricecode',
       '/transaction/subfile/detail/detail.account',
       '/transaction/subfile/detail/detail.costprice',
       '/transaction/subfile/detail/detail.description',
       '/transaction/subfile/detail/detail.discount',
       '/transaction/subfile/detail/detail.gross',
       '/transaction/subfile/detail/detail.net',
       '/transaction/subfile/detail/detail.orderqty',
       '/transaction/subfile/detail/detail.saleunit',
       '/transaction/subfile/detail/detail.stockcode',
       '/transaction/subfile/detail/detail.stockqty',
       '/transaction/subfile/detail/detail.tax',
       '/transaction/subfile/detail/detail.taxcode',
       '/transaction/subfile/detail/detail.unitprice', '/transaction/theirref',
      

In [28]:
# Ensure key types match
df["/transaction/#id"] = df["/transaction/#id"].astype(str)
df_rules["/transaction/#id"] = df_rules["/transaction/#id"].astype(str)

# Merge safely
merged = df.merge(
    df_rules,
    on="/transaction/#id",
    how="left",
    suffixes=("", "_rule")
)

# Confirm month exists
assert "month" in merged.columns

scorecard = (
    merged.groupby("month", as_index=False)
    .agg(
        total_transactions=("/transaction/#id", "nunique"),
        total_lines=("/transaction/subfile/detail/detail.stockcode", "count"),
        gross_sales=("/transaction/gross", "sum"),
        completeness_stockcode=(
            "/transaction/subfile/detail/detail.stockcode",
            lambda s: s.notna().mean()
        ),
        completeness_net=(
            "/transaction/subfile/detail/detail.net",
            lambda s: s.notna().mean()
        ),
        rule_gross_equals_net=("rule_pass", "mean"),
    )
)

In [29]:
# Convert proportions
percent_cols = [
    "completeness_stockcode",
    "completeness_net",
    "rule_gross_equals_net"
]

scorecard[percent_cols] = (
    scorecard[percent_cols] * 100
).round(2)

scorecard.to_excel("Cactus_2025_DQ_scorecard.xlsx", index=False)